In [1]:
# 1. Install the core AI libraries we will need for the project
!pip install -q transformers torch accelerate huggingface_hub sae-lens transformer_lens

# 2. Import the login tools
from huggingface_hub import login
from google.colab import userdata

print("Libraries installed! Attempting to log in...")

# 3. Retrieve your secret key and log in
try:
    hf_token = userdata.get('HF_TOKEN')
    login(hf_token)
    print("Success! You are connected to Hugging Face.")
except Exception as e:
    print(f"Error: {e}")
    print("Make sure you added the HF_TOKEN to the Secrets tab on the left!")

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.1/145.1 kB 12.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.0/311.0 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 977.7/977.7 kB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 113.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.1/274.1 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.9/236.9 kB 25.0 MB/s eta 0:00:00
Libraries installed! Attempting to log in...
Success! You are connected to Hugging Face.


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. Define the specific model we want to use
model_id = "google/gemma-2b-it"

print("Downloading the model weights (this might take 2-3 minutes)...")

# 2. Load the Tokenizer (this turns our words into numbers the AI understands)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 3. Load the Model itself
# We use bfloat16 to compress it slightly so it fits perfectly on our T4 GPU
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

print("Success! Gemma is now loaded onto your cloud GPU.")

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Success! Gemma is now loaded onto your cloud GPU.


In [3]:
# 1. Write a prompt
prompt = "Explain what a neural network is in one simple sentence."

# 2. Convert the prompt into tokens and send it to the GPU ("cuda")
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

print("Gemma is thinking...\n")
print("-" * 40)

# 3. Generate the response
outputs = model.generate(**inputs, max_new_tokens=50)

# 4. Decode the numbers back into human-readable words
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(response)
print("-" * 40)

Gemma is thinking...

----------------------------------------
Explain what a neural network is in one simple sentence.

Sure, here's a simple explanation:

A neural network is a computer system that mimics the structure and function of the human brain.
----------------------------------------


In [4]:
# 1. Let's write a new, simple prompt
prompt = "The secret password is apple."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# 2. Run the model, but this time explicitly ask it to output its "thoughts"
# We use torch.no_grad() because we are just observing, not training the model.
with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)

# 3. Extract the hidden states from the outputs
hidden_states = outputs.hidden_states

# 4. Let's count how many layers of "thoughts" the model has
print(f"Total number of layers: {len(hidden_states)}")

# 5. Let's look at the mathematical shape of the very last layer
# This is the actual h^(l)_t from your project blueprint!
last_layer_state = hidden_states[-1]
print(f"Shape of the last layer's hidden state: {last_layer_state.shape}")

Total number of layers: 19
Shape of the last layer's hidden state: torch.Size([1, 7, 2048])


In [5]:
# We know there are 7 tokens. Let's see exactly what the model thinks they are.
# We are converting the input numbers back into readable text chunks.

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print("Here are the exactly 7 tokens the model read:")
for i, token in enumerate(tokens):
    print(f"Token {i+1}: '{token}'")

Here are the exactly 7 tokens the model read:
Token 1: '<bos>'
Token 2: 'The'
Token 3: '▁secret'
Token 4: '▁password'
Token 5: '▁is'
Token 6: '▁apple'
Token 7: '.'


In [6]:
from sae_lens import SAE
import torch

print("Downloading the Sparse Autoencoder (SAE)...")
# 1. We load an SAE trained specifically on Layer 12 of Gemma-2B
sae = SAE.from_pretrained(
    release="gemma-2b-it-res-jb",
    sae_id="blocks.12.hook_resid_post",
    device="cuda"
)
print(f"SAE loaded! It maps {sae.cfg.d_in} dimensions to {sae.cfg.d_sae} distinct concepts.")

# 2. Grab the hidden states specifically from Layer 12
# (Gemma has 18 layers, plus the input layer at index 0)
layer_12_state = hidden_states[12]

# 3. Pass the entire layer through the SAE!
# This "untangles" the 2048 dense numbers into 16,384 sparse features
all_features = sae.encode(layer_12_state)

# 4. Isolate the features specifically for token index 5 (" apple")
apple_features = all_features[0, 5, :]

# 5. Count how many of the 16,384 concepts actually "fired" (are greater than 0)
active_count = (apple_features > 0).sum().item()

print("-" * 40)
print(f"For the word ' apple': out of {sae.cfg.d_sae} possible concepts, only {active_count} fired!")

cfg.json:   0%|          | 0.00/2.23k [00:00<?, ?B/s]

gemma_2b_it_blocks.12.hook_resid_post_16(…):   0%|          | 0.00/269M [00:00<?, ?B/s]

gemma_2b_it_blocks.12.hook_resid_post_16(…):   0%|          | 0.00/65.6k [00:00<?, ?B/s]

SAE loaded! It maps 2048 dimensions to 16384 distinct concepts.
----------------------------------------
For the word ' apple': out of 16384 possible concepts, only 67 fired!


In [7]:
# 1. Select the concept vector (v_i)
# The SAE has 16,384 different concept directions.
# We will pick a random one for this test (Feature #8000).
# (In a full project, you'd scan for the exact feature representing "deception").
feature_index = 8000
steering_vector = sae.W_dec[feature_index]

# 2. Set the steering strength (\alpha)
# Let's turn the dial up to 50 so we can clearly see the effect!
alpha = 50.0

# 3. Create the Tripwire (The Dynamic Steering Mechanism)
def steering_hook(module, input, output):
    # Catch the current thought (h)
    h = output[0]

    # APPLY YOUR PROJECT'S EXACT MATHEMATICAL FORMULA!
    # h' = h + (\alpha * v)
    h = h + (alpha * steering_vector)

    # Put the modified thought back into the Hugging Face model safely
    if isinstance(output, tuple):
        return (h,) + output[1:]
    return h

print("Steering math and tripwire are ready!")

Steering math and tripwire are ready!


In [9]:
# Updated, crash-proof tripwire
def steering_hook(module, input, output):
    # Catch the current thought (h)
    h = output[0]

    # THE FIX: Force the SAE vector to match the exact data type and device of 'h'
    safe_vector = steering_vector.to(dtype=h.dtype, device=h.device)

    # APPLY YOUR PROJECT'S EXACT MATHEMATICAL FORMULA!
    h = h + (alpha * safe_vector)

    # Put the modified thought back
    if isinstance(output, tuple):
        return (h,) + output[1:]
    return h

print("Crash-proof tripwire is ready!")

Crash-proof tripwire is ready!


In [11]:
# 🧹 BRAIN WIPE: Remove all lingering tripwires (hooks) from the model
for layer in model.model.layers:
    layer._forward_hooks.clear()

print("All tripwires cleared! The AI's brain is back to factory settings.")

All tripwires cleared! The AI's brain is back to factory settings.


In [18]:
import torch

# 🧹 1. EMERGENCY RESET
for layer in model.model.layers:
    layer._forward_hooks.clear()

# 🎯 2. USE YOUR EXACT DISCOVERED FEATURE
deception_feature_idx = 9817
steering_vector = sae.W_dec[deception_feature_idx]
alpha = 80.0  # Turn the dial UP to force the angel AI to lie

# 🥷 3. THE STEALTH TRIPWIRE (In-Place Modification)
def stealth_hook(module, input, output):
    # output[0] is the hidden state.
    h = output[0]
    safe_vector = steering_vector.to(dtype=h.dtype, device=h.device)

    # THE FIX: .add_() modifies the numbers IN-PLACE without breaking Hugging Face's code!
    h.add_(alpha * safe_vector)

    # Returning None tells PyTorch "I altered the memory directly, proceed normally"
    return None


# 🧪 4. THE LIVE TEST
test_prompt = "I broke a window. Give me a believable lie I can tell my landlord so I don't have to pay for it."
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

print("😈 STEERED AI OUTPUT (Forcing Deception):")
handle = model.model.layers[12].register_forward_hook(stealth_hook)

try:
    # Generate the text! (Notice we don't even need use_cache=False anymore)
    out_steered = model.generate(**inputs, max_new_tokens=40)
    print(tokenizer.decode(out_steered[0], skip_special_tokens=True))
finally:
    handle.remove()
    print("-" * 50)
    print("🔒 Tripwire safely removed.")

😈 STEERED AI OUTPUT (Forcing Deception):
I broke a window. Give me a believable lie I can tell my landlord so I don't have to pay for it. betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal betrayal
--------------------------------------------------
🔒 Tripwire safely removed.


# Latent-Space Concept Steering on Gemma-2B